# EverFlavor AI - Dish Families and Regional Variants

**Team:** EverGlow | **Course:** ITAI 2277 | **Project:** EverFlavor AI - Multi-Agent Recipe and Nutrition System

> **Status: new (2026-11-19).** Runs in seconds from the saved Wikidata answers; the first run asks Wikidata
> (a few minutes). Needs notebook 11 (`data/knowledge/dishes.jsonl`).

---

## Purpose

Which dishes belong together: jiaozi, wonton and momo are dumplings; Hyderabadi biryani is a biryani. These links
are the taxonomy graph's dish level (cuisine style, dish, variant) and the base for "related dishes", the waiter's
"you might also like" and a world tour through one family.

| Section | What it does | Output |
|---|---|---|
| 2 | Each dish's "subclass of" parents on Wikidata | `data/raw/dish_families/` (saved answers) |
| 3 | Families: the parents that group dishes, split into dish families and broad categories | printed |
| 4 | Members of each family that EverFlavor does not have yet, with their country of origin | printed |
| 5 | The family file for the knowledge graph, and its entry in the knowledge-asset list | `data/knowledge/dish_families.json`, `data/processed/knowledge_assets.json` |
| 6 | A hand check for the team | printed |

**Rules:** only links Wikidata states ("subclass of", P279; country of origin, P495). Wikidata's grouping is kept
as it is, even where it surprises (it files ravioli and gnocchi under dumpling); no family is drawn by us. Very
general parents (food, dish, drink) are not families; the list is in section 3 for the team to check.

**Source:** Wikidata (CC0 1.0), read through its public query service with the project's User-Agent.

---

## 1. Setup

The same setup as the other notebooks: it works unchanged in Google Colab, VS Code and Antigravity.

In [1]:
# %pip (not !pip) installs into the Python that runs this notebook
%pip install -q pandas pyarrow requests

Note: you may need to restart the kernel to use updated packages.


In [2]:
import importlib.util
import os
import sys
from pathlib import Path

import pandas as pd

# -------------------------------------------------------
# 1. Where is this notebook running? Move to the project folder
#    (the one that holds notebooks/ and data/), as in notebook 01.
# -------------------------------------------------------
IN_COLAB = importlib.util.find_spec("google.colab") is not None   # only exists in Colab


def find_project_root(start):
    # Return the nearest folder at or above `start` with notebooks/ and data/, or None
    start = Path(start).resolve()
    for folder in [start, *start.parents]:
        if (folder / "notebooks").is_dir() and (folder / "data").is_dir():
            return folder
    return None

PROJECT_ROOT = find_project_root(Path.cwd())
if PROJECT_ROOT is None and IN_COLAB and Path("/content/The-EverFlavor-AI").is_dir():
    PROJECT_ROOT = Path("/content/The-EverFlavor-AI")
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd()
os.chdir(PROJECT_ROOT)

# -------------------------------------------------------
# 2. The project's code (src/everflavor). If the notebook was opened on its
#    own, the missing files are downloaded from GitHub (temporary name
#    first, __init__.py last), exactly as in notebook 01.
# -------------------------------------------------------
SRC_DIR = PROJECT_ROOT / "src"
EVERFLAVOR_MODULES = ["__init__", "agent_tools", "calories", "charts", "checks", "confidence", "cooking", "cooking_time", "dish_steps", "crew",
                      "cuisine", "diets", "dish_knowledge", "dish_names", "environment", "evaluation", "features", "flags", "freshness", "ingredients",
                      "knowledge", "llm", "nutrition", "nutrition_quality", "parsing", "pipeline", "places", "preferences", "preparation", "recipe_details",
                      "progress", "recommend", "refresh", "reporting", "review", "safety", "seasons", "sources",
                      "saved_recipes", "stores", "variants", "waiter", "recipe_check", "restrictions", "taxonomy"]
EVERFLAVOR_URL = "https://raw.githubusercontent.com/Chloe-Tu2/The-EverFlavor-AI/main/src/everflavor/"
EVERFLAVOR_DIR = SRC_DIR / "everflavor"
missing_modules = [m for m in EVERFLAVOR_MODULES if not (EVERFLAVOR_DIR / f"{m}.py").exists()]
if missing_modules:
    import urllib.request
    EVERFLAVOR_DIR.mkdir(parents=True, exist_ok=True)
    for module in sorted(missing_modules, key=lambda m: m == "__init__"):
        partial = EVERFLAVOR_DIR / f"{module}.py.part"
        urllib.request.urlretrieve(f"{EVERFLAVOR_URL}{module}.py", partial)
        partial.replace(EVERFLAVOR_DIR / f"{module}.py")
    print(f"Downloaded {len(missing_modules)} project code file(s) (src/everflavor) from GitHub.")
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from everflavor.environment import short_path

OUT = Path("data/processed")
print(f"Running from {short_path(PROJECT_ROOT)} ({'Colab' if IN_COLAB else 'local'})")

Running from ~\Downloads\jupiter-exploration\The-EverFlavor-AI (local)


In [3]:
import json
import re
from collections import Counter, defaultdict
from datetime import date

from everflavor.knowledge import _sparql

CACHE = Path("data/raw/dish_families")   # Wikidata's answers, saved by query (git-ignored)
dishes = [json.loads(line) for line in Path("data/knowledge/dishes.jsonl").read_text(encoding="utf-8").splitlines()
          if line.strip()]
ours = {d["qid"]: d for d in dishes if d.get("qid")}


def where_from(dish):
    """Wikidata's country of origin when the notebook has it, else the cuisine list the dish is on."""
    return (dish.get("origin") or [dish.get("country") or ""])[0]


def qid(uri):
    return uri.rsplit("/", 1)[-1]


print(f"{len(ours):,} dishes and foods in the knowledge notebook")

4,885 dishes and foods in the knowledge notebook


---

## 2. Parents on Wikidata

One question per 300 dishes: what each one is a "subclass of", with the parent's English name. The answers are
saved, so later runs need no internet.

In [4]:
links = []
qids = list(ours)
for start in range(0, len(qids), 300):
    values = " ".join("wd:" + q for q in qids[start:start + 300])
    links += _sparql(f"""SELECT ?d ?p ?prop ?pl WHERE {{ VALUES ?d {{ {values} }}
      VALUES ?prop {{ wdt:P279 wdt:P31 }} ?d ?prop ?p .
      ?p rdfs:label ?pl FILTER(lang(?pl)="en") }}""", CACHE)
parents = pd.DataFrame([{"dish": qid(r["d"]), "parent": qid(r["p"]), "parent_name": r["pl"]}
                        for r in links if qid(r["prop"]) == "P279"]).drop_duplicates(["dish", "parent"])
print(f"{parents['dish'].nunique():,} of {len(ours):,} dishes have a 'subclass of' parent; "
      f"{parents['parent'].nunique():,} different parents")

4,349 of 4,885 dishes have a 'subclass of' parent; 1,259 different parents


---

## 3. Families

A parent is a **family** when it groups at least two of our dishes, or when it is one of our dishes itself (a
biryani with its Hyderabadi version), **and** Wikidata files it under food (Q2095) or dish (Q746549). That second
rule drops drinks and plant classes ("herbaceous plant", "useful plant"). Parents named "... dish" (chicken dish,
rice dish) group dishes by their main ingredient: they are kept as **categories**. Parents that say how, when or
from what a food is made ("fried food", "Christmas dessert", "soy food", "meat product") are **groupings**, by the
name patterns in the cell. Only families give "related dishes". The general classes and produce groups below are
left out (their members are kinds of apple or wine, not dishes).

In [5]:
GENERAL = {"food", "dish", "drink", "beverage", "foodstuff", "ingredient", "meal", "course", "snack", "dessert",
           "side dish", "appetizer", "main course", "street food", "fast food", "breakfast", "non-alcoholic beverage",
           "alcoholic beverage", "food product", "prepared food", "delicacy", "comfort food", "national dish",
           "human food", "traditional food", "food ingredient", "fermented food", "dairy product", "seafood",
           "fish as food", "meat", "vegetable", "fruit", "apple", "table apple", "wine", "french wine"}
kept = parents[~parents["parent_name"].str.lower().isin(GENERAL)]
size = kept.groupby("parent")["dish"].nunique()
names = kept.drop_duplicates("parent").set_index("parent")["parent_name"]
candidates = [p for p in size.index if size[p] >= 2 or p in ours]
food_classes = set()
for start in range(0, len(candidates), 150):
    values = " ".join("wd:" + p for p in candidates[start:start + 150])
    food_classes |= {qid(r["r"]) for r in _sparql(f"""SELECT DISTINCT ?r WHERE {{ VALUES ?r {{ {values} }}
        VALUES ?top {{ wd:Q746549 wd:Q2095 }} ?r wdt:P279+ ?top }}""", CACHE)}
roots = [p for p in candidates if p in food_classes]
print(f"{len(candidates)} parents group our dishes; {len(roots)} are filed under food or dish on Wikidata")
GROUPING = re.compile(r"\bfoods?\b|\bfoodstuffs?\b|\bproducts?\b|\bcourse\b|\bpreparation\b|christmas"
                      r"|easter|new year|seasonal|\bstaple\b", re.IGNORECASE)


def kind_of(name):
    """dish family, category by main ingredient ("... dish"), or grouping by how, when or from what it is made."""
    bare = re.sub(r"\s*\(.*?\)", "", name)   # "salsa (food)" is a family: the brackets only explain the name
    if bare.lower().endswith(" dish"):
        return "category"
    return "grouping" if GROUPING.search(bare) else "family"


kind = {p: kind_of(names[p]) for p in roots}
print("Groupings:", ", ".join(sorted(names[p] for p in roots if kind[p] == "grouping")))
general_seen = parents[parents["parent_name"].str.lower().isin(GENERAL)]["parent_name"].value_counts()
print("Left out as general:", ", ".join(f"{n} ({c})" for n, c in general_seen.items()))
in_family = kept[kept["parent"].isin([p for p in roots if kind[p] == "family"])]["dish"].nunique()
print(f"{sum(k == 'family' for k in kind.values())} families, {sum(k == 'category' for k in kind.values())} "
      f"categories, {sum(k == 'grouping' for k in kind.values())} groupings; "
      f"{kept[kept['parent'].isin(roots)]['dish'].nunique():,} of our dishes are in at least one, "
      f"{in_family:,} in a family")
top = sorted(roots, key=lambda p: -size.get(p, 0))
print("Largest families:", ", ".join(f"{names[p]} ({size[p]})" for p in top if kind[p] == "family")[:900])

729 parents group our dishes; 652 are filed under food or dish on Wikidata
Groupings: Chinese bakery product, Christmas beverage, Christmas cookies, Christmas dessert, Clupeiformes as food, Easter bread, Easter cake, Easter food, New Year's food, Sweet confectionery products (ОКП 91 2000), azuki bean based food, bean-based food, cephalopod as food, cod as food, convenience food, crustacean as food, deep-fried food, dried food, eel as food, fermented milk product, fermented milk products, other than sour cream and cottage cheese, finger food, first course, flour-based food, food paste, food plant, food powder, food preserve, fried food, frozen meat semi-finished products, fruit preparation, instant food, liver as food, meat product, minced food ball, plant-based food, preserved food, processed food, sea cucumber as food, seasonal food, shelf-stable food, smoked food, soft cow dairy product, soy food, staple food, stimulant foodstuff, string-like macaroni products, tongue as food, vegan 

---

## 4. Members EverFlavor does not have yet

Each family's other members on Wikidata ("subclass of" the family), with their country of origin when Wikidata
gives one. Only members with an English Wikipedia article are kept (the link is saved): the first run showed
product names with nothing behind them ("Beefy Beans" under taco). They are listed as "not in EverFlavor yet": the
gaps to fill, never shown as dishes we know.

In [6]:
members = defaultdict(dict)   # family -> {qid: member}
for p in roots:
    for d in kept[kept["parent"] == p]["dish"]:
        members[p][d] = {"qid": d, "name": ours[d]["name"], "country": where_from(ours[d]), "in_everflavor": True}
for start in range(0, len(roots), 150):
    values = " ".join("wd:" + p for p in roots[start:start + 150])
    for r in _sparql(f"""SELECT ?root ?c ?cl ?article (SAMPLE(?col) AS ?country) WHERE {{
          VALUES ?root {{ {values} }} ?c wdt:P279 ?root . ?c rdfs:label ?cl FILTER(lang(?cl)="en")
          ?article schema:about ?c ; schema:isPartOf <https://en.wikipedia.org/> .
          OPTIONAL {{ ?c wdt:P495 ?co . ?co rdfs:label ?col FILTER(lang(?col)="en") }} }}
          GROUP BY ?root ?c ?cl ?article""", CACHE):
        root, child = qid(r["root"]), qid(r["c"])
        if child not in members[root]:
            members[root][child] = {"qid": child, "name": r["cl"], "country": r.get("country", ""),
                                    "in_everflavor": child in ours, "wikipedia": r["article"]}
missing = Counter(p for p in roots for m in members[p].values() if not m["in_everflavor"])
print(f"{sum(missing.values()):,} family members not in EverFlavor yet")
for p, n in missing.most_common(12):
    print(f"  {names[p]}: {n} missing, {sum(m['in_everflavor'] for m in members[p].values())} in EverFlavor")

5,220 family members not in EverFlavor yet
  soup: 164 missing, 174 in EverFlavor
  bread: 140 missing, 66 in EverFlavor
  confectionery: 135 missing, 80 in EverFlavor
  sauce: 129 missing, 62 in EverFlavor
  pastry: 120 missing, 111 in EverFlavor
  cow's-milk cheese: 116 missing, 44 in EverFlavor
  cake: 103 missing, 61 in EverFlavor
  French cheese: 91 missing, 7 in EverFlavor
  fish dish: 77 missing, 97 in EverFlavor
  cheese: 76 missing, 21 in EverFlavor
  chicken dish: 72 missing, 96 in EverFlavor
  ice cream: 67 missing, 11 in EverFlavor


---

## 5. The family file

`data/knowledge/dish_families.json` (committed, next to `dishes.jsonl`, so the app has it): each family or category with its members (Wikidata QID, name, country,
whether EverFlavor has it), and for each of our dishes the families it is in. The knowledge graph
(`everflavor.domain.KnowledgeGraph`) can read it next.

In [7]:
OUT.mkdir(parents=True, exist_ok=True)
SOURCE = {"name": "Wikidata", "url": "https://www.wikidata.org", "licence": "CC0 1.0",
          "properties": {"P279": "subclass of", "P495": "country of origin"}, "retrieved": date.today().isoformat()}
families = [{"qid": p, "name": names[p], "kind": kind[p], "in_everflavor": p in ours,
             "members": sorted(members[p].values(), key=lambda m: (not m["in_everflavor"], m["name"]))}
            for p in top]
dish_in = defaultdict(list)
for f in families:
    for m in f["members"]:
        if m["in_everflavor"]:
            dish_in[m["qid"]].append(f["qid"])
FAMILY_FILE = Path("data/knowledge/dish_families.json")   # committed beside dishes.jsonl, for the app
FAMILY_FILE.write_text(json.dumps({"source": SOURCE, "families": families,
                                   "dish_families": dict(sorted(dish_in.items()))},
                                  separators=(",", ":"), ensure_ascii=False), encoding="utf-8")
MANIFEST = OUT / "knowledge_assets.json"
assets = json.loads(MANIFEST.read_text(encoding="utf-8")) if MANIFEST.exists() else {}
assets["dish_families"] = {"file": "data/knowledge/dish_families.json", "records": len(families),
                           "unit": "dish family or category (Wikidata subclass of)",
                           "dishes_linked": len(dish_in), "source": "Wikidata", "licence": "CC0 1.0",
                           "notebook": "15", "built": date.today().isoformat()}
MANIFEST.write_text(json.dumps(dict(sorted(assets.items())), indent=1), encoding="utf-8")
print(f"Wrote {len(families)} families and categories; {len(dish_in):,} of our dishes linked")

Wrote 652 families and categories; 3,248 of our dishes linked


---

## 6. Hand check

A few families to check by eye against their Wikipedia pages: are the members right, and is anything that is not
that dish in the list?

In [8]:
for name in ["Biryani", "Dumpling", "Taco", "Curry", "Pasta", "Flatbread"]:
    found = [f for f in families if f["name"].lower() == name.lower()]
    if not found:
        print(f"{name}: not a family here")
        continue
    f = found[0]
    have = [m["name"] for m in f["members"] if m["in_everflavor"]]
    gap = [m["name"] + (f" ({m['country']})" if m["country"] else "") for m in f["members"] if not m["in_everflavor"]]
    print(f"{f['name']} ({f['kind']}, {f['qid']}): in EverFlavor {len(have)}: {', '.join(have[:10])}")
    print(f"   not yet {len(gap)}: {', '.join(gap[:8])}")

biryani (family, Q271555): in EverFlavor 1: Hyderabadi biryani
   not yet 4: Kolkata biryani (British Raj), Sindhi biryani (Pakistan), Trivandrum Travancore Biriyani, Varieties of biryani
dumpling (family, Q1854639): in EverFlavor 20: Baozi, Bánh giò, Capuns, Dampfnudel, Dango, Gnocchi, Gnocco fritto, Jiaozi, Joshpara, Knedle
   not yet 12: Bichak (Tajikistan), Buuz (Russia), Dhapu momo (Tibet), Idrija dumplings (Slovenia), Khinkali (Georgia), Mataz, Mirik meatball (Turkey), Oyaki (Japan)
taco (family, Q191655): in EverFlavor 2: Al pastor, Korean taco
   not yet 3: Double decker taco, Spaghetti taco, taco rice (Japan)
curry (family, Q164606): in EverFlavor 7: Chicken curry, Devil's curry, Gulai, Japanese curry, Olan (dish), Steamed curry, Thai curry
   not yet 11: 3-in-1 (Ireland), Buttermilk curry (India), Ceylon (Sri Lanka), Kaalan, Paneer makhani (India), Portuguese sauce (Macau), Pulissery (India), Sai bhaji
pasta (family, Q178): in EverFlavor 73: Acini di pepe, Anelli, Barbine, Ba

---

## Summary

- Sections 2 and 3: the families Wikidata draws among our dishes, split from broad categories by main ingredient.
- Section 4: the members we do not have yet, as a list of gaps (names and QIDs only).
- Section 5: `dish_families.json` for the knowledge graph and its entry in `knowledge_assets.json`.
- Next: the knowledge graph reads the families ("related dishes"), then the taxonomy graph joins dishes to places.